# 0413 · RandomForest (ensamble), versión 2

> **Versión 2 (05-11-2026, evaluación del director).** Lee `V4/MODELADO/` (0411 v2, cinco bloques de abril a agosto) y comprueba la huella de la base antes de entrenar (A6). Random Forest no tiene función de pérdida en la búsqueda (criterio de error cuadrático); se compara con las referencias de media.

**Qué hace.** Entrena y valida solo RandomForest, en las dos tareas:

| Tarea | Qué predice | Métrica para elegir |
| --- | --- | --- |
| `clf` | Probabilidad de incumplir el SLA | AP (mayor es mejor) |
| `reg` | Horas de última milla | MAE en horas (menor es mejor) |

**Random Forest** promedia muchos árboles de decisión entrenados con muestras distintas de guías y de variables. Es el ensamble más simple de los tres y el más lento de predecir con muchas filas.

**Pasos.** 1) carga lo que dejó 0411; 2) búsqueda aleatoria de hiperparámetros con validación de origen móvil; 3) validación con la mejor configuración en los tres bloques (abril, mayo, junio); 4) vista rápida de sus métricas; 5) guarda predicciones, importancias y parámetros para 0416.

**No depende de los otros cuadernos de modelos.** Solo necesita que 0411 se haya corrido. La prueba final no se toca aquí.

**Tiempo.** La búsqueda hace `N_CONFIGS_BUSQUEDA × 3 bloques × 2 tareas` ajustes sobre una submuestra de `MAX_FILAS_BUSQUEDA` filas. Para una primera corrida rápida, bajar `N_CONFIGS_BUSQUEDA` a 3.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 116.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 185.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 19.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 149.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, pliegues_origen_movil, partes_del_bloque, marco_predicciones,
                                       construir_modelo, predecir, muestrear_configs, puntaje_config, mejores, params_de,
                                       importancia_pliegue, tabla_metricas, verificar_huella,
                                       PERDIDA_CORTA, mejores_por_perdida)

In [0]:
MODELO = "RandomForest"
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"

N_CONFIGS_BUSQUEDA = 20        # configuraciones aleatorias por tarea
MAX_FILAS_BUSQUEDA = 300_000   # submuestra de entrenamiento en la búsqueda (no en la validación)
EJECUTAR_BUSQUEDA  = True      # False: usa los parámetros que ya guardó este cuaderno
N_PERMUTACION      = 10_000    # filas de validación para la importancia por permutación en cada bloque

In [0]:
# Hiperparámetros de base (si no hay búsqueda) y espacio de búsqueda registrado (realimentación P4)
PARAMS_BASE = dict(n_estimators=200, min_samples_leaf=20, max_features="sqrt", max_samples=0.2)
ESPACIO = dict(n_estimators=[100, 200, 300], max_depth=[None, 12, 20], min_samples_leaf=[1, 5, 20, 50],
               max_features=["sqrt", 0.3, 0.5], max_samples=[0.1, 0.2, 0.3])

## 1. Cargar la preparación de 0411

Base de modelado, historia sin futuro de cada bloque y configuración común (periodos, semilla, variables). Los pliegues se vuelven a armar con la misma función del asesor.

In [0]:
print("Huella de la base comprobada:", verificar_huella(CARPETA_MODELADO)[:16], "...")   # A6
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · preparado el {CFG['fecha_preparacion']}")

Huella de la base comprobada: dc7400e56c1f74c7 ...
1,553,980 guías · versiones: ['internas', 'internas+clima'] · preparado el 2026-11-07T14:07:13.019713


## 2. Búsqueda aleatoria de hiperparámetros (solo en validación)

Se prueban `N_CONFIGS_BUSQUEDA` configuraciones de `ESPACIO`. Cada una se entrena en cada bloque con una submuestra de hasta `MAX_FILAS_BUSQUEDA` filas y se evalúa en el bloque completo. Se elige la de mejor promedio entre bloques. Si la base trae la versión «internas+clima_reajustada», se busca también para ella.

In [0]:
def buscar(version, numericas):
    """Prueba todas las configuraciones de una versión de variables; una fila por configuración."""
    filas = []
    for tarea, y_col, _ in TAREAS:
        for k, cfg in enumerate(muestrear_configs(ESPACIO, N_CONFIGS_BUSQUEDA, SEMILLA + len(MODELO))):
            if tarea == "clf":
                cfg = {a: b for a, b in cfg.items() if a != "objetivo_reg"}   # solo aplica a regresión
            vals = puntaje_config(tarea, MODELO, cfg, k, PLIEGUES[tarea], HIST, df, y_col,
                                  CATEGORICAS, numericas, MAX_FILAS_BUSQUEDA, SEMILLA)
            filas.append({"version": version, "tarea": tarea, "modelo": MODELO, "config": k,
                          "metrica": "AP" if tarea == "clf" else "MAE_h", "promedio": np.mean(vals),
                          "por_bloque": [round(v, 4) for v in vals], **{f"p_{a}": b for a, b in cfg.items()}})
            print(f"{version} {tarea} config {k}: {np.mean(vals):.4f}")
    return pd.DataFrame(filas)

In [0]:
RUTA_PARAMS = f"{CARPETA_MODELADO}parametros_{MODELO}.json"
t_ini = time.time()
if EJECUTAR_BUSQUEDA:
    versiones_busqueda = ["internas"] + [v for v in VERSIONES if v == "internas+clima_reajustada"]
    tabla_busqueda = pd.concat([buscar(v, VERSIONES[v]) for v in versiones_busqueda], ignore_index=True)
    tabla_busqueda.to_pickle(f"{CARPETA_MODELADO}busqueda_{MODELO}.pkl")       # exacta, para 0416
    tabla_busqueda.to_csv(f"{CARPETA_MODELADO}busqueda_{MODELO}.csv", index=False)
    MEJORES = mejores(tabla_busqueda, {MODELO: ESPACIO})
else:
    MEJORES = json.load(open(RUTA_PARAMS))["mejores"]
print(f"Búsqueda: {time.time() - t_ini:.0f} s")
display(pd.DataFrame(MEJORES).T)

internas clf config 0: 0.3153
internas clf config 1: 0.3199
internas clf config 2: 0.3249
internas clf config 3: 0.3221
internas clf config 4: 0.3209
internas clf config 5: 0.2982
internas clf config 6: 0.3250
internas clf config 7: 0.3076
internas clf config 8: 0.3182
internas clf config 9: 0.3097
internas clf config 10: 0.3278
internas clf config 11: 0.3292
internas clf config 12: 0.3205
internas clf config 13: 0.3127
internas clf config 14: 0.3263
internas clf config 15: 0.3193
internas clf config 16: 0.3283
internas clf config 17: 0.3231
internas clf config 18: 0.3188
internas clf config 19: 0.3203
internas reg config 0: 8.5132
internas reg config 1: 7.7418
internas reg config 2: 7.8651
internas reg config 3: 7.6040
internas reg config 4: 7.8604
internas reg config 5: 9.7496
internas reg config 6: 7.9448
internas reg config 7: 10.0400
internas reg config 8: 8.5992
internas reg config 9: 9.4621
internas reg config 10: 7.6883
internas reg config 11: 7.7701
internas reg config 12: 7.5

n_estimators,max_depth,min_samples_leaf,max_features,max_samples
300.0,12.0,5.0,0.3,0.1
100.0,null,50.0,0.3,0.1


## 3. Validación de origen móvil con la mejor configuración

En cada versión de variables, tarea y bloque: se entrena con todo lo anterior al bloque, se predice el bloque y se guardan las predicciones. Con las variables internas se mide además la importancia por permutación (insumo de 0420).

In [0]:
pred, imp = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        params = params_de(MEJORES, {MODELO: PARAMS_BASE}, version, tarea, MODELO)
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            m = construir_modelo(tarea, MODELO, numericas, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            pred.append(marco_predicciones(version, tarea, ini, Dva, yva).assign(modelo=MODELO, pred=predecir(m, Xva, tarea)))
            if version == "internas":
                etiquetas = {"tarea": tarea, "modelo": MODELO, "bloque": ini.date()}
                imp.append(importancia_pliegue(m, Xva, yva, tarea, N_PERMUTACION, SEMILLA, etiquetas))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred, imp = pd.concat(pred, ignore_index=True), pd.concat(imp, ignore_index=True)

internas · clf · 2026-04-01 listo (62 s)
internas · clf · 2026-05-01 listo (144 s)
internas · clf · 2026-06-01 listo (245 s)
internas · clf · 2026-07-01 listo (376 s)
internas · clf · 2026-08-01 listo (544 s)
internas · reg · 2026-04-01 listo (578 s)
internas · reg · 2026-05-01 listo (615 s)
internas · reg · 2026-06-01 listo (667 s)
internas · reg · 2026-07-01 listo (724 s)
internas · reg · 2026-08-01 listo (798 s)
internas+clima · clf · 2026-04-01 listo (864 s)
internas+clima · clf · 2026-05-01 listo (953 s)
internas+clima · clf · 2026-06-01 listo (1066 s)
internas+clima · clf · 2026-07-01 listo (1210 s)
internas+clima · clf · 2026-08-01 listo (1384 s)
internas+clima · reg · 2026-04-01 listo (1411 s)
internas+clima · reg · 2026-05-01 listo (1453 s)
internas+clima · reg · 2026-06-01 listo (1504 s)
internas+clima · reg · 2026-07-01 listo (1568 s)
internas+clima · reg · 2026-08-01 listo (1655 s)


## 4. Vista rápida de RandomForest

Solo para revisar este modelo. La comparación con los demás se hace en 0416.

In [0]:
display(tabla_metricas(pred, ["version", "tarea", "modelo"]).round(4))
display(tabla_metricas(pred, ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,RandomForest,966041,0.0765,0.3151,0.7232,0.0621,0.2363,null,null,null,null,null,null
internas,reg,RandomForest,958475,null,null,null,null,null,7.5064,31.6337,9.8827,40.2168,104.0422,0.97
internas+clima,clf,RandomForest,966041,0.0765,0.3142,0.7225,0.0621,0.2367,null,null,null,null,null,null
internas+clima,reg,RandomForest,958475,null,null,null,null,null,7.6577,31.6223,10.4758,39.9774,109.2663,0.97


version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,RandomForest,2026-04-01,193214,0.0711,0.3681,0.7434,0.0543,0.2127,null,null,null,null,null,null
internas,clf,RandomForest,2026-05-01,204833,0.0857,0.3651,0.7461,0.0664,0.2482,null,null,null,null,null,null
internas,clf,RandomForest,2026-06-01,185344,0.082,0.3231,0.7069,0.0669,0.2547,null,null,null,null,null,null
internas,clf,RandomForest,2026-07-01,246075,0.0715,0.2776,0.7067,0.0612,0.2351,null,null,null,null,null,null
internas,clf,RandomForest,2026-08-01,136575,0.0717,0.3083,0.7532,0.0616,0.2294,null,null,null,null,null,null
internas,reg,RandomForest,2026-04-01,191924,null,null,null,null,null,6.9326,37.1401,8.3043,36.6966,100.44,0.9666
internas,reg,RandomForest,2026-05-01,202331,null,null,null,null,null,7.4565,34.2903,9.6714,41.9526,95.3176,0.9647
internas,reg,RandomForest,2026-06-01,183841,null,null,null,null,null,8.1473,37.9066,9.8586,50.9264,88.3052,0.9648
internas,reg,RandomForest,2026-07-01,244689,null,null,null,null,null,7.814,24.6139,10.6039,36.9578,122.0958,0.9775
internas,reg,RandomForest,2026-08-01,135690,null,null,null,null,null,6.9696,18.0251,10.291,33.8215,110.4141,0.9763


In [0]:
# Variables con más peso en validación (promedio de los bloques)
display(imp.groupby(["tarea", "variable"]).importancia.mean().sort_values(ascending=False).groupby(level=0).head(8).round(4))

tarea  variable            
reg    categoria_zona          0.6898
       hist_h_zona_franja      0.6674
       hist_inc_zona_franja    0.3674
       margen_sla_en_t0_h      0.3397
       hist_inc_cliente        0.3028
       ID_PRODUCTO             0.2501
       hist_h_cliente          0.2497
       n_hist_cliente          0.2488
clf    margen_sla_en_t0_h      0.1011
       dia_semana              0.0597
       categoria_zona          0.0330
       hist_h_zona_franja      0.0282
       hist_inc_zona_franja    0.0263
       ID_PRODUCTO             0.0106
       n_hist_cliente          0.0083
       hist_inc_cliente        0.0076
Name: importancia, dtype: float64

## 5. Guardar para 0416

In [0]:
pred.to_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO}.parquet", index=False)
imp.to_parquet(f"{CARPETA_MODELADO}importancia_{MODELO}.parquet", index=False)
with open(RUTA_PARAMS, "w") as f:
    json.dump({"modelo": MODELO, "espacio": ESPACIO, "params_base": PARAMS_BASE, "semilla": SEMILLA,
               "n_configs": N_CONFIGS_BUSQUEDA, "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES}, f, indent=2, default=str)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/MODELADO/
